# The previous flat methods

Earlier client versions exposed one method per operation on the client itself:
`client.get_images(...)`, `client.upsert_group(...)`, `client.create_dataset(...)`.
Code written that way keeps working - each old name is the same call, with the same
parameters and the same return value, under the current spelling. `help()` on either
shows the same signature.

New code should use the grouped spelling from the other notebooks. This notebook is
the map between the two, plus a short end-to-end flow written entirely the old way.

| Old | Now |
|---|---|
| **images** | |
| `client.get_images(...)` | `client.images.list(...)` |
| `client.get_images_iter(...)` | `client.images.iter(...)` |
| `client.get_random_images(...)` | `client.images.random(...)` |
| `client.count_images(...)` | `client.images.count(...)` |
| `client.get_image(...)` | `client.images.get(...)` |
| `client.create_image(...)` | `client.images.create(...)` |
| `client.create_images(...)` | `client.images.create_many(...)` |
| `client.update_image(...)` | `client.images.update(...)` |
| `client.update_images(...)` | `client.images.update_many(...)` |
| `client.delete_image(...)` | `client.images.delete(...)` |
| `client.get_image_similarity(...)` | `client.images.similarity(...)` |
| `client.get_similar_images(...)` | `client.images.similar(...)` |
| `client.get_related_images(...)` | `client.images.related(...)` |
| `client.delete_image_latent(...)` | `client.images.delete_latent(...)` |
| `client.aggregate_images(...)` | `client.images.aggregate(...)` |
| `client.bucket_images(...)` | `client.images.bucket(...)` |
| `client.get_image_groups(...)` | `client.images.groups(...)` |
| **tags** | |
| `client.get_tags(...)` | `client.tags.list(...)` |
| `client.get_tag(...)` | `client.tags.get(...)` |
| `client.create_tag(...)` | `client.tags.create(...)` |
| `client.tag_images(...)` | `client.tags.apply(...)` |
| **roles** | |
| `client.get_roles(...)` | `client.roles.list(...)` |
| `client.create_role(...)` | `client.roles.create(...)` |
| `client.delete_role(...)` | `client.roles.delete(...)` |
| **group_types** | |
| `client.get_group_types(...)` | `client.group_types.list(...)` |
| `client.get_group_type(...)` | `client.group_types.get(...)` |
| `client.create_group_type(...)` | `client.group_types.create(...)` |
| `client.delete_group_type(...)` | `client.group_types.delete(...)` |
| **groups** | |
| `client.get_groups(...)` | `client.groups.list(...)` |
| `client.get_group(...)` | `client.groups.get(...)` |
| `client.update_group(...)` | `client.groups.update(...)` |
| `client.delete_group(...)` | `client.groups.delete(...)` |
| `client.get_group_members(...)` | `client.groups.members(...)` |
| `client.upsert_group(...)` | `client.groups.upsert(...)` |
| `client.create_groups(...)` | `client.groups.create_many(...)` |
| `client.delete_groups(...)` | `client.groups.delete_many(...)` |
| **queries** | |
| `client.get_queries(...)` | `client.queries.list(...)` |
| `client.get_query(...)` | `client.queries.get(...)` |
| `client.create_query(...)` | `client.queries.create(...)` |
| `client.update_query(...)` | `client.queries.update(...)` |
| **datasets** | |
| `client.get_datasets(...)` | `client.datasets.list(...)` |
| `client.get_dataset(...)` | `client.datasets.get(...)` |
| `client.create_dataset(...)` | `client.datasets.create(...)` |
| `client.update_dataset(...)` | `client.datasets.update(...)` |
| `client.delete_dataset(...)` | `client.datasets.delete(...)` |
| `client.add_dataset_groups(...)` | `client.datasets.add_groups(...)` |
| `client.remove_dataset_groups(...)` | `client.datasets.remove_groups(...)` |
| `client.add_dataset_images(...)` | `client.datasets.add_images(...)` |
| `client.freeze_dataset(...)` | `client.datasets.freeze(...)` |
| `client.unfreeze_dataset(...)` | `client.datasets.unfreeze(...)` |
| `client.copy_dataset(...)` | `client.datasets.copy(...)` |

## Connect

In [1]:
import os
import uuid

from dataroom_client import DataRoomClientSync, DataRoomError

os.environ["DATAROOM_API_KEY"] = 'YOUR_KEY_HERE'
os.environ["DATAROOM_API_URL"] = 'http://localhost:8000/api/'

client = DataRoomClientSync()
client.get_images(limit=1)

# Everything below carries this suffix, so re-runs never collide.
RUN = uuid.uuid4().hex[:6]
print('connected; run id:', RUN)

connected; run id: c40a0d


## Images

In [2]:
images = client.get_images(limit=3, fields=['id'])
print('get_images   :', [img['id'] for img in images])
print('count_images :', client.count_images())
print('get_image    :', client.get_image(images[0]['id'], fields=['id', 'source']))

get_images   : ['bench_0_0', 'bench_0_1', 'bench_0_10']
count_images : 103010
get_image    : {'id': 'bench_0_0', 'source': 'bulk_update'}


## Groups

The same `garment_pair` type the use-cases notebook builds - creating it here too keeps
this notebook self-contained, and both spellings hit the same rows.

In [3]:
for role, description in [
    ('on_hang', 'Garment on a hanger or mannequin, no model.'),
    ('on_model_front', 'The same garment worn by a model, shot from the front.'),
    ('detail_shot', 'Optional close-up of fabric or print.'),
]:
    try:
        client.create_role(name=role, description=description)
        print('created role', role)
    except DataRoomError:
        print('role exists', role)

try:
    client.create_group_type(
        name='garment_pair',
        description='One garment: the hanger shot paired with the on-model shot.',
        metadata_schema={'type': 'object', 'additionalProperties': True},
        roles=[
            {'role': 'on_hang',        'is_required': True},
            {'role': 'on_model_front', 'is_required': True},
            {'role': 'detail_shot',    'is_required': False},
        ],
    )
    print('created group_type garment_pair')
except DataRoomError:
    print('group_type exists garment_pair')

role exists on_hang


role exists on_model_front
role exists detail_shot
group_type exists garment_pair


In [4]:
image_ids = [img['id'] for img in client.get_images(limit=4, fields=['id'])]

group = client.upsert_group(
    name=f'legacy-{RUN}-01',
    type='garment_pair',
    metadata={'sku': f'{RUN}-01'},
    members=[
        {'image_id': image_ids[0], 'role': 'on_hang'},
        {'image_id': image_ids[1], 'role': 'on_model_front'},
    ],
)
print('upsert_group      :', group['id'])
for m in client.get_group_members(group['id']):
    print(f"get_group_members :  {m['role']:16s} {m['image_id']}")
print('get_group         :', client.get_group(group['id'])['name'])

upsert_group      : 76c97124-5f29-4684-8ec0-ac379e5321c3
get_group_members :  on_model_front   bench_0_1
get_group_members :  on_hang          bench_0_0
get_group         : legacy-c40a0d-01


## Datasets

In [5]:
ds = client.create_dataset(name='Legacy flow', slug=f'legacy-{RUN}', type='garment_pair')
SV = ds['slug_version']
client.add_dataset_groups(SV, [group['id']])
client.freeze_dataset(SV)
print('create_dataset + add_dataset_groups + freeze_dataset:',
      SV, '| group_count =', client.get_dataset(SV)['group_count'])
for d in client.get_datasets(slug=f'legacy-{RUN}'):
    print('get_datasets(slug=...):', d['slug_version'])

create_dataset + add_dataset_groups + freeze_dataset: legacy-c40a0d/1 | group_count = 1
get_datasets(slug=...): legacy-c40a0d/1


## Cleanup

In [6]:
for d in client.get_datasets(search=RUN):
    client.delete_dataset(d['slug_version'])
    print('delete_dataset:', d['slug_version'])

stale = [g['id'] for g in client.get_groups(search=f'legacy-{RUN}', limit=1000)]
print('delete_groups :', client.delete_groups(stale)['deleted_count'] if stale else 0)

delete_dataset: legacy-c40a0d/1


delete_groups : 1
